In [ ]:
import psycopg2
from datetime import datetime

In [ ]:
DB = {
    "host": "rawg-db.cf0ec26s6wkm.eu-north-1.rds.amazonaws.com",
    "database": "postgres",
    "user": "rawg_admin",
    "password": "rawg2026-daniel",
    "port": 5432,
    "connect_timeout": 15
}

In [ ]:
def connect():
    return psycopg2.connect(**DB)


conn = connect()
cur = conn.cursor()

print("Connected to RDS")

In [ ]:
START = datetime.utcnow()

print("Schema initialization started at:", START)

In [ ]:
cur.execute("""
DO $$
DECLARE r RECORD;
BEGIN
 FOR r IN (
     SELECT tablename
     FROM pg_tables
     WHERE schemaname = 'public'
 )
 LOOP
  EXECUTE 'DROP TABLE IF EXISTS '
          || quote_ident(r.tablename)
          || ' CASCADE';
 END LOOP;
END $$;
""")

conn.commit()

print("Previous schema cleared")

In [ ]:
cur.execute("""

-- =========================
-- CORE ENTITIES
-- =========================

CREATE TABLE games (

    game_id BIGINT PRIMARY KEY,

    name TEXT,
    slug TEXT,

    release_date DATE,
    playtime INT,

    background_image TEXT
);


CREATE TABLE platforms (

    platform_id BIGINT PRIMARY KEY,
    name TEXT
);


CREATE TABLE tags (

    tag_id BIGINT PRIMARY KEY,
    name TEXT
);

""")

conn.commit()

print("Core tables created")

In [ ]:
cur.execute("""

-- =========================
-- GAME METRICS (HISTORICAL)
-- =========================

CREATE TABLE game_metrics (

    game_id BIGINT,
    snapshot_date DATE,

    rating FLOAT,
    ratings_count INT,
    reviews_count INT,

    metacritic INT,
    added INT,

    community_rating FLOAT,

    PRIMARY KEY (game_id, snapshot_date),

    FOREIGN KEY (game_id)
        REFERENCES games(game_id)
        ON DELETE CASCADE
);

""")

conn.commit()

print("Metrics table created")

In [ ]:
cur.execute("""

-- =========================
-- RELATION TABLES
-- =========================

CREATE TABLE game_platforms (

    game_id BIGINT,
    platform_id BIGINT,

    PRIMARY KEY (game_id, platform_id),

    FOREIGN KEY (game_id)
        REFERENCES games(game_id),

    FOREIGN KEY (platform_id)
        REFERENCES platforms(platform_id)
);


CREATE TABLE game_tags (

    game_id BIGINT,
    tag_id BIGINT,

    PRIMARY KEY (game_id, tag_id),

    FOREIGN KEY (game_id)
        REFERENCES games(game_id),

    FOREIGN KEY (tag_id)
        REFERENCES tags(tag_id)
);

""")

conn.commit()

print("Bridge tables created")

In [ ]:
cur.execute("""

-- =========================
-- ETL MONITORING
-- =========================

CREATE TABLE etl_runs (

    run_id SERIAL PRIMARY KEY,

    pipeline_name TEXT,

    started_at TIMESTAMP,
    finished_at TIMESTAMP,

    rows_processed INT,
    bad_files INT,

    status TEXT,
    error_message TEXT
);

""")

conn.commit()

print("ETL control table created")

In [ ]:
cur.execute("""

-- =========================
-- PERFORMANCE INDEXES
-- =========================

CREATE INDEX idx_metrics_game
ON game_metrics(game_id);


CREATE INDEX idx_metrics_date
ON game_metrics(snapshot_date);


CREATE INDEX idx_games_name
ON games(name);


CREATE INDEX idx_tags_name
ON tags(name);

""")

conn.commit()

print("Indexes created")

In [ ]:
cur.execute("""
SELECT tablename
FROM pg_tables
WHERE schemaname='public'
ORDER BY tablename;
""")

tables = cur.fetchall()

print("\nTables created:\n")

for t in tables:
    print("-", t[0])

In [ ]:
END = datetime.utcnow()

cur.close()
conn.close()

print("\nSchema initialization completed")
print("Start:", START)
print("End  :", END)
print("Duration (min):", round((END-START).total_seconds()/60,2))